In [1]:
import pandas as pd
print(pd.__version__)

3.0.6


In [5]:
import pandas as pd

df=pd.read_csv("../data/raw/dvf_2024_33.csv.gz", low_memory=False)
df.shape

(82927, 40)

In [6]:
df.head()

,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
0,2024-363954,2024-01-03,1,Vente,336120.0,16.0,NaN,IMP DU PETIT FEYDIEU,0801,33290.0,...,Maison,92.0,4.0,S,sols,NaN,NaN,500.0,-0.625239,44.974353
1,2024-363955,2024-01-08,1,Vente,400000.0,NaN,NaN,CHE DE LEOGNAN,6946,33610.0,...,Maison,110.0,4.0,S,sols,NaN,NaN,444.0,-0.637939,44.747139
2,2024-363956,2024-01-05,1,Vente,175000.0,328.0,NaN,CRS DE LA SOMME,8705,33800.0,...,Appartement,53.0,2.0,NaN,NaN,NaN,NaN,NaN,-0.572046,44.816789
3,2024-363956,2024-01-05,1,Vente,175000.0,2.0,NaN,RUE PIERRE DE LADIME,7298,33800.0,...,Dépendance,NaN,0.0,NaN,NaN,NaN,NaN,NaN,-0.572046,44.816789
4,2024-363957,2024-01-08,1,Vente,128000.0,28.0,NaN,RUE ANDRE MOULINAT,0012,33480.0,...,NaN,NaN,NaN,AB,terrains a bâtir,NaN,NaN,522.0,-0.814988,45.028059


In [7]:
df.columns.tolist()


['id_mutation',
 'date_mutation',
 'numero_disposition',
 'nature_mutation',
 'valeur_fonciere',
 'adresse_numero',
 'adresse_suffixe',
 'adresse_nom_voie',
 'adresse_code_voie',
 'code_postal',
 'code_commune',
 'nom_commune',
 'code_departement',
 'ancien_code_commune',
 'ancien_nom_commune',
 'id_parcelle',
 'ancien_id_parcelle',
 'numero_volume',
 'lot1_numero',
 'lot1_surface_carrez',
 'lot2_numero',
 'lot2_surface_carrez',
 'lot3_numero',
 'lot3_surface_carrez',
 'lot4_numero',
 'lot4_surface_carrez',
 'lot5_numero',
 'lot5_surface_carrez',
 'nombre_lots',
 'code_type_local',
 'type_local',
 'surface_reelle_bati',
 'nombre_pieces_principales',
 'code_nature_culture',
 'nature_culture',
 'code_nature_culture_speciale',
 'nature_culture_speciale',
 'surface_terrain',
 'longitude',
 'latitude']

In [8]:
df["id_mutation"].nunique()


29429

In [9]:
df["nature_mutation"].value_counts() 

nature_mutation
Vente                                 76020
Vente en l'état futur d'achèvement     5662
Echange                                 957
Adjudication                            178
Vente terrain à bâtir                    92
Expropriation                            18
Name: count, dtype: int64

In [10]:
df["type_local"].value_counts(dropna=False)

type_local
NaN                                         36304
Dépendance                                  19984
Maison                                      14371
Appartement                                  9729
Local industriel. commercial ou assimilé     2539
Name: count, dtype: int64

In [11]:
df.isna().mean().sort_values(ascending=False).head(15)

ancien_code_commune             1.000000
ancien_id_parcelle              1.000000
ancien_nom_commune              1.000000
lot5_surface_carrez             0.999928
lot4_surface_carrez             0.999855
lot5_numero                     0.999337
lot4_numero                     0.998854
lot3_surface_carrez             0.997878
numero_volume                   0.997540
lot3_numero                     0.993114
lot2_surface_carrez             0.987386
lot2_numero                     0.962967
adresse_suffixe                 0.951355
lot1_surface_carrez             0.908220
code_nature_culture_speciale    0.907220
dtype: float64

In [12]:
df[["valeur_fonciere", "surface_reelle_bati"]].describe()

,valeur_fonciere,surface_reelle_bati
count,8.237100e+04,26357.000000
mean,8.959978e+05,116.907311
std,3.768020e+06,701.244566
min,1.000000e+00,1.000000
25%,1.060000e+05,51.000000
50%,2.219500e+05,78.000000
75%,3.946750e+05,107.000000
max,6.860122e+07,64565.000000


# 01 - Cleaning DVF data, Gironde (33), 2024
**Goal:** build a clean table of residential sales (one row = one sale) with `price_per_m2`.
**Source:** DVF géolocalisées (Etalab), data.gouv.fr.
**Method:** apply one filter at a time and record the row count after each, to build a cleaning funnel.

### Step 1 - Raw data
82,927 rows and 40 columns, but only 29,429 distinct sales (`id_mutation`).
One sale can span several rows (apartment + cellar + parking), and the total price is repeated on each row.

In [13]:
funnel = {"1. Raw rows": len(df)}
funnel

{'1. Raw rows': 82927}

### Step 2 - Keep `nature_mutation == "Vente"`
Removed exchanges, auctions, expropriations and sales of new builds (VEFA).
**Why:** these are not ordinary market prices. VEFA could be analysed separately later.
**Rows after:** X

In [14]:
d = df[df["nature_mutation"] == "Vente"].copy()
funnel["2. Nature = Vente"] = len(d)
funnel

{'1. Raw rows': 82927, '2. Nature = Vente': 76020}

### Step 3 - Remove sales that include commercial/industrial premises
**Why:** the price covers non-residential space, so price per m² would be wrong.
**Rows after:** X

In [15]:
mixed = d.loc[d["type_local"] == "Local industriel. commercial ou assimilé", "id_mutation"].unique()
d = d[~d["id_mutation"].isin(mixed)]
funnel["3. No commercial premises"] = len(d)
funnel

{'1. Raw rows': 82927,
 '2. Nature = Vente': 76020,
 '3. No commercial premises': 69872}

### Step 4 - Keep `Maison` and `Appartement`
Dropped dependencies (cellars, parkings) and bare land.
**Why:** the question is about residential price per m².
**Rows after:** X

In [17]:
res = d[d["type_local"].isin(["Maison", "Appartement"])].copy()
funnel["4. Houses/apartments rows"] = len(res)
funnel

{'1. Raw rows': 82927,
 '2. Nature = Vente': 76020,
 '3. No commercial premises': 69872,
 '4. Houses/apartments rows': 22897}

### Step 5 - Drop duplicate rows of the same property
A property on several cadastral parcels appears several times with the same surface and price.
**Why:** otherwise one sale would count several times.
**Rows after:** X

In [18]:
res = res.drop_duplicates(subset=["id_mutation", "type_local", "surface_reelle_bati", "nombre_pieces_principales"])
funnel["5. After dropping duplicates"] = len(res)
funnel

{'1. Raw rows': 82927,
 '2. Nature = Vente': 76020,
 '3. No commercial premises': 69872,
 '4. Houses/apartments rows': 22897,
 '5. After dropping duplicates': 20126}

### Step 6 - Keep sales with exactly one house or apartment
**Why:** if a sale includes several units, the total price cannot be split fairly between them.
**Trade-off:** this loses some data, but it keeps price per m² reliable.
**Rows after:** X

In [19]:
n = res.groupby("id_mutation")["id_mutation"].transform("size")
res = res[n == 1].copy()
funnel["6. One property per sale"] = len(res)
funnel

{'1. Raw rows': 82927,
 '2. Nature = Vente': 76020,
 '3. No commercial premises': 69872,
 '4. Houses/apartments rows': 22897,
 '5. After dropping duplicates': 20126,
 '6. One property per sale': 18238}

### Step 7 - Require price and surface, then compute `price_per_m2`
`price_per_m2 = valeur_fonciere / surface_reelle_bati`
**Why:** the ratio can't be computed without both.
**Rows after:** X (final count before outlier removal)

In [20]:
res = res.dropna(subset=["valeur_fonciere", "surface_reelle_bati"]).copy()
res["price_per_m2"] = res["valeur_fonciere"] / res["surface_reelle_bati"]
funnel["7. Price and surface present"] = len(res)
pd.Series(funnel)

1. Raw rows                     82927
2. Nature = Vente               76020
3. No commercial premises       69872
4. Houses/apartments rows       22897
5. After dropping duplicates    20126
6. One property per sale        18238
7. Price and surface present    18228
dtype: int64

### Step 8 - Look at the distribution of `price_per_m2`
Before removing outliers, I looked at the percentiles.
- Median: about 3,415 €/m² (apartments 3,673, houses 3,138).
- The extremes are clearly errors: the minimum is 0.006 €/m² (a 1 € sale on a large surface) and the maximum is about 107,000 €/m².
- The 1st percentile is about 490 €/m² and the 99th about 11,995 €/m², so the real market sits in between.
**Decision:** thresholds are set in Step 10, based on these numbers and not on guesses.

In [21]:
res["price_per_m2"].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99])

count     18228.000000
mean       3716.429277
std        2643.100454
min           0.006250
1%          490.173077
5%         1169.811321
25%        2446.493380
50%        3414.634146
75%        4400.000000
95%        6959.257389
99%       11994.829787
max      107333.333333
Name: price_per_m2, dtype: float64

In [22]:
res.groupby("type_local")["price_per_m2"].median()

type_local
Appartement    3673.238636
Maison         3137.978348
Name: price_per_m2, dtype: float64

### Step 10 - Outlier removal
Rules (chosen after looking at the percentiles):
- `valeur_fonciere >= 10,000`: removes symbolic sales (e.g. 1 euro transfers).
- `9 <= surface_reelle_bati <= 500`: 9 m² is the smallest legal dwelling, above 500 m² is not a normal home.
- `500 <= price_per_m2 <= 15,000`: below/above are almost certainly data errors (1st / 99.5th percentile region).

**Removed:** X rows (X%). **Rows after:** X.
**Check:** I inspected the removed rows to confirm they were errors, not real expensive/cheap sales.
**Limitation:** the thresholds are a judgment call. They are fixed for all communes, so a few real sales in prime areas may be cut.
**What I saw in the removed rows:** (write 1-2 sentences, e.g. "the lowest rows are 1 € or symbolic sales, the highest are tiny surfaces with normal prices, so these are data errors").

In [23]:
before = len(res)

mask = (
    (res["valeur_fonciere"] >= 10_000)          # removes symbolic sales (family transfers at 1 euro, etc.)
    & res["surface_reelle_bati"].between(9, 500) # 9 m2 = smallest legal dwelling; above 500 m2 is not a normal home
    & res["price_per_m2"].between(500, 15_000)   # around the 1st and 99.5th percentiles
)

removed = res[~mask]
clean = res[mask].copy()

funnel["8. After outlier removal"] = len(clean)
print(f"Removed {before - len(clean)} rows ({(before - len(clean)) / before:.1%})")
pd.Series(funnel)

Removed 293 rows (1.6%)


1. Raw rows                     82927
2. Nature = Vente               76020
3. No commercial premises       69872
4. Houses/apartments rows       22897
5. After dropping duplicates    20126
6. One property per sale        18238
7. Price and surface present    18228
8. After outlier removal        17935
dtype: int64

In [24]:
removed[["valeur_fonciere", "surface_reelle_bati", "price_per_m2", "type_local", "nom_commune"]].sort_values("price_per_m2").head(10)

,valeur_fonciere,surface_reelle_bati,price_per_m2,type_local,nom_commune
46244,1.0,160.0,0.006250,Maison,Sainte-Eulalie
5459,1.0,146.0,0.006849,Appartement,Bordeaux
5670,1.0,114.0,0.008772,Appartement,Bordeaux
45618,1.0,107.0,0.009346,Maison,Les Artigues-de-Lussac
32708,1.0,98.0,0.010204,Maison,Martillac
34448,1.0,84.0,0.011905,Appartement,Tresses
67002,1.0,83.0,0.012048,Maison,Carignan-de-Bordeaux
7038,1.0,17.0,0.058824,Appartement,Bègles
43066,50.0,95.0,0.526316,Maison,Fargues
1728,500.0,136.0,3.676471,Maison,Bruges


In [25]:
removed[["valeur_fonciere", "surface_reelle_bati", "price_per_m2", "type_local", "nom_commune"]].sort_values("price_per_m2").tail(10)

,valeur_fonciere,surface_reelle_bati,price_per_m2,type_local,nom_commune
31679,2216530.0,54.0,41046.851852,Maison,Lège-Cap-Ferret
10791,4461409.5,94.0,47461.803191,Maison,Saint-Médard-en-Jalles
40474,2600000.0,52.0,50000.000000,Maison,Lège-Cap-Ferret
34466,5970000.0,116.0,51465.517241,Maison,Lège-Cap-Ferret
54780,11332550.0,200.0,56662.750000,Maison,La Teste-de-Buch
67750,8000000.0,139.0,57553.956835,Maison,Lège-Cap-Ferret
77037,4619805.5,80.0,57747.568750,Maison,Le Barp
64615,5400000.0,89.0,60674.157303,Maison,Gujan-Mestras
43994,28100000.0,379.0,74142.480211,Maison,La Teste-de-Buch
73250,16100000.0,150.0,107333.333333,Maison,Lège-Cap-Ferret


In [26]:
clean["price_per_m2"].describe(percentiles=[.01, .25, .5, .75, .99])

count    17935.000000
mean      3627.120362
std       1790.687521
min        500.000000
1%         735.571895
25%       2475.000000
50%       3421.052632
75%       4390.289920
99%      10178.583104
max      15000.000000
Name: price_per_m2, dtype: float64

### Step 11 - Bordeaux check
The question is about Bordeaux, so I compared it with the whole of Gironde.
- Number of clean sales in Bordeaux: X (apartments X, houses X).
- Median price/m²: apartments X, houses X, versus 3,673 and 3,138 for all of Gironde.
**Note:** the sample size matters. Breakdowns with few sales (for example houses in Bordeaux) are less reliable.

In [27]:
clean["nom_commune"].value_counts().head(10)

nom_commune
Bordeaux             3705
Mérignac              881
Pessac                585
Talence               524
Villenave-d'Ornon     410
Arcachon              380
Bègles                349
La Teste-de-Buch      346
Le Bouscat            323
Eysines               306
Name: count, dtype: int64

In [28]:
clean[clean["nom_commune"] == "Bordeaux"].groupby("type_local")["price_per_m2"].agg(["count", "median"])

,count,median
type_local,,
Appartement,2844,4218.750000
Maison,861,4952.631579


### Step 12 - Save the cleaned data
Saved to `data/processed/dvf_2024_33_clean.csv`. This folder is in `.gitignore`, so the file stays local. It will be loaded into PostgreSQL later.

In [29]:
import os
os.makedirs("../data/processed", exist_ok=True)
clean.to_csv("../data/processed/dvf_2024_33_clean.csv", index=False)